In [1]:
import sys
import os

sys.path.append(os.path.abspath("../src"))

In [2]:
import pandas as pd

df = pd.read_csv("../data/raw/car_rental_raw.csv")

df.head()

,Reservation_ID,Customer_ID,Vehicle_ID,Vehicle_Class,Booking_TS,Pickup_TS,Return_TS,City,Pickup_Lat,Pickup_Lon,...,Harsh_Events,Damage_Flag,Rate,Promo_Code,GST_Amount,Total_Amount,Rate_Plan,Driver_License,Notes,Customer_Feedback
0,RES-00001,C1434,car-18,Luxury,2026-03-22 13:01,23-03-2026 03:01,2026-03-24 18:01,Kolkata,22.576220,88.345171,...,3,0,1500/day,BOGUS99,75.0,1575.0,standard,INVALID,Scratch on windows,Good mileage
1,RES-00002,C2584,CAR-12,SUV,2026-02-25 11:17,2026/02/27 18:17,2026-03-01 14:17,delhi,28.690319,77.120789,...,2,0,800,NaN,NaN,840.0,standard,INVALID,Bad Seats Quality,Good mileage
2,RES-00003,C2307,car-28,Economy,2026-02-26 18:07,2026-02-28 17:07:00,2026-03-04 00:07,Chennai,13.090883,80.252533,...,5,0,2500,BOGUS99,NaN,2625.0,eco,DL12345,scratch on door,Good mileage
3,RES-00004,C9751,car-24,Compact,2026-02-04 11:40,05-02-2026 14:40,2026-02-09 12:40,kolkata,22.580673,88.371242,...,4,0,2500,BOGUS99,125.0,2625.0,eco,INVALID,scratch on door,Smooth going
4,RES-00005,C6155,CAR-07,Luxury,2026-02-18 00:59,18/02/2026 08:59,2026-02-21 23:59,Mumbai,19.088202,72.873747,...,1,0,1200/day,BOGUS99,NaN,1260.0,premium,DL12345,Speakers not working,Smooth going


In [3]:
from cleaning import cleaner
from processing import deduplicator
from validation import validator

In [4]:
df["Vehicle_ID"] = df["Vehicle_ID"].apply(cleaner.clean_vehicle_id)
df['Vehicle_ID'].head()

0    CAR-18
1    CAR-12
2    CAR-28
3    CAR-24
4    CAR-07
Name: Vehicle_ID, dtype: str

In [5]:
df["Pickup_TS"] = df["Pickup_TS"].apply(cleaner.clean_timestamp)
df["Return_TS"] = df["Return_TS"].apply(cleaner.clean_timestamp)
df[["Pickup_TS", "Return_TS"]].head()

,Pickup_TS,Return_TS
0,2026-03-23 03:01,2026-03-24 18:01
1,2026-02-27 18:17,2026-03-01 14:17
2,2026-02-28 17:07,2026-03-04 00:07
3,2026-02-05 14:40,2026-02-09 12:40
4,2026-02-18 08:59,2026-02-21 23:59


In [6]:
df.columns

Index(['Reservation_ID', 'Customer_ID', 'Vehicle_ID', 'Vehicle_Class',
       'Booking_TS', 'Pickup_TS', 'Return_TS', 'City', 'Pickup_Lat',
       'Pickup_Lon', 'Drop_Lat', 'Drop_Lon', 'Payment', 'Odo_Start', 'Odo_End',
       'Fuel_Level', 'GPS_Lat', 'GPS_Lon', 'Max_Speed_kmh', 'Harsh_Events',
       'Damage_Flag', 'Rate', 'Promo_Code', 'GST_Amount', 'Total_Amount',
       'Rate_Plan', 'Driver_License', 'Notes', 'Customer_Feedback'],
      dtype='str')

In [7]:
df[["Odo_Start", "Odo_End"]].head()

,Odo_Start,Odo_End
0,"30,179km","30,857 km"
1,41908km,41829
2,"100,504 km","101,678km"
3,"66,049km","67,079 km"
4,"41,765km",42647km


In [8]:
df['Odo_Start'] = df['Odo_Start'].apply(cleaner.clean_odometer)
df['Odo_End'] = df['Odo_End'].apply(cleaner.clean_odometer)

df[["Odo_Start", "Odo_End"]].head()

,Odo_Start,Odo_End
0,30179,30857
1,41908,41829
2,100504,101678
3,66049,67079
4,41765,42647


In [9]:
df['Fuel_Level'].head()

0    93 %
1    90 %
2     94%
3      7%
4    0.40
Name: Fuel_Level, dtype: str

In [10]:
df['Fuel_Level'] = df['Fuel_Level'].apply(cleaner.clean_fuel_level)
df['Fuel_Level'].head()

0    0.93
1    0.90
2    0.94
3    0.07
4    0.40
Name: Fuel_Level, dtype: float64

In [11]:
df['Rate'].head()

0    1500/day
1         800
2        2500
3        2500
4    1200/day
Name: Rate, dtype: str

In [12]:
df['Rate'] = df['Rate'].apply(cleaner.clean_rate)
df['Rate'].head()

0    1500
1     800
2    2500
3    2500
4    1200
Name: Rate, dtype: int64

In [13]:
df['City'].head()

0    Kolkata
1      delhi
2    Chennai
3    kolkata
4     Mumbai
Name: City, dtype: str

In [14]:
df['City'] = df['City'].apply(cleaner.clean_city)
df['City'].head()

0    Kolkata
1      Delhi
2    Chennai
3    Kolkata
4     Mumbai
Name: City, dtype: str

In [15]:
df["Reservation_ID"].duplicated().sum()

np.int64(280)

In [16]:
records = df.to_dict(orient="records")

unique_records, duplicates_count = deduplicator.remove_duplicates(records)

print("Duplicates removed:", duplicates_count)

df = pd.DataFrame(unique_records)

df.head()

Duplicates removed: 280


,Reservation_ID,Customer_ID,Vehicle_ID,Vehicle_Class,Booking_TS,Pickup_TS,Return_TS,City,Pickup_Lat,Pickup_Lon,...,Harsh_Events,Damage_Flag,Rate,Promo_Code,GST_Amount,Total_Amount,Rate_Plan,Driver_License,Notes,Customer_Feedback
0,RES-00001,C1434,CAR-18,Luxury,2026-03-22 13:01,2026-03-23 03:01,2026-03-24 18:01,Kolkata,22.576220,88.345171,...,3,0,1500,BOGUS99,75.0,1575.0,standard,INVALID,Scratch on windows,Good mileage
1,RES-00002,C2584,CAR-12,SUV,2026-02-25 11:17,2026-02-27 18:17,2026-03-01 14:17,Delhi,28.690319,77.120789,...,2,0,800,NaN,NaN,840.0,standard,INVALID,Bad Seats Quality,Good mileage
2,RES-00003,C2307,CAR-28,Economy,2026-02-26 18:07,2026-02-28 17:07,2026-03-04 00:07,Chennai,13.090883,80.252533,...,5,0,2500,BOGUS99,NaN,2625.0,eco,DL12345,scratch on door,Good mileage
3,RES-00004,C9751,CAR-24,Compact,2026-02-04 11:40,2026-02-05 14:40,2026-02-09 12:40,Kolkata,22.580673,88.371242,...,4,0,2500,BOGUS99,125.0,2625.0,eco,INVALID,scratch on door,Smooth going
4,RES-00005,C6155,CAR-07,Luxury,2026-02-18 00:59,2026-02-18 08:59,2026-02-21 23:59,Mumbai,19.088202,72.873747,...,1,0,1200,BOGUS99,NaN,1260.0,premium,DL12345,Speakers not working,Smooth going


In [17]:
# df[df["Reservation_ID"].duplicated(keep=False)]

In [18]:
# df = df.drop_duplicates(subset="Reservation_ID", keep="first")

In [19]:
df["Reservation_ID"].duplicated().sum()

np.int64(0)

In [20]:
df.columns

Index(['Reservation_ID', 'Customer_ID', 'Vehicle_ID', 'Vehicle_Class',
       'Booking_TS', 'Pickup_TS', 'Return_TS', 'City', 'Pickup_Lat',
       'Pickup_Lon', 'Drop_Lat', 'Drop_Lon', 'Payment', 'Odo_Start', 'Odo_End',
       'Fuel_Level', 'GPS_Lat', 'GPS_Lon', 'Max_Speed_kmh', 'Harsh_Events',
       'Damage_Flag', 'Rate', 'Promo_Code', 'GST_Amount', 'Total_Amount',
       'Rate_Plan', 'Driver_License', 'Notes', 'Customer_Feedback'],
      dtype='str')

In [21]:
validation_results = df.apply(validator.validate_timestamps, axis=1)

df["timestamp_valid"] = validation_results.apply(lambda x: x[0])
df["timestamp_error"] = validation_results.apply(lambda x: x[1])

In [22]:
# show invalid rows 
df[df["timestamp_valid"] == False][["Pickup_TS","Return_TS","timestamp_error"]]

,Pickup_TS,Return_TS,timestamp_error
33,NaN,2026-02-05 15:27,Unparseable timestamp(s)
171,NaN,2026-01-11 21:06,Unparseable timestamp(s)
554,NaN,2026-03-27 08:25,Unparseable timestamp(s)
693,NaN,2026-01-30 20:56,Unparseable timestamp(s)
788,NaN,2026-02-07 07:07,Unparseable timestamp(s)
...,...,...,...
6599,NaN,2026-03-06 05:45,Unparseable timestamp(s)
6645,NaN,2026-02-22 00:50,Unparseable timestamp(s)
6715,NaN,2026-01-17 22:21,Unparseable timestamp(s)
6814,NaN,2026-03-11 21:37,Unparseable timestamp(s)


In [23]:
#count valid vs invalid 
invalid_ts = df[df["timestamp_valid"] == False]

print("Total records:", len(df))
print("Invalid timestamps:", len(invalid_ts))

invalid_ts.head()

Total records: 7000
Invalid timestamps: 64


,Reservation_ID,Customer_ID,Vehicle_ID,Vehicle_Class,Booking_TS,Pickup_TS,Return_TS,City,Pickup_Lat,Pickup_Lon,...,Rate,Promo_Code,GST_Amount,Total_Amount,Rate_Plan,Driver_License,Notes,Customer_Feedback,timestamp_valid,timestamp_error
33,RES-00034,C7455,CAR-19,Economy,2026-01-27 04:27,NaN,2026-02-05 15:27,Kolkata,22.586220,88.380682,...,1200,BOGUS99,NaN,1260.0,premium,DL67543,Scratch on windows,good service,False,Unparseable timestamp(s)
171,RES-00172,C3604,CAR-28,Economy,2026-01-04 14:06,NaN,2026-01-11 21:06,Bengaluru,12.959693,77.602144,...,1200,SAVE10,60.0,1260.0,standard,DL90871,Bad Seats Quality,Good speaker,False,Unparseable timestamp(s)
554,RES-00555,C2468,CAR-25,Luxury,2026-03-20 08:25,NaN,2026-03-27 08:25,Delhi,28.710212,77.097515,...,2000,BOGUS99,100.0,2100.0,eco,DL90871,Contact Rahul 9876543210,email test@gmail.com,False,Unparseable timestamp(s)
693,RES-00694,C3735,CAR-03,Luxury,2026-01-24 19:56,NaN,2026-01-30 20:56,Chennai,13.092414,80.255756,...,2500,DISC15,125.0,2625.0,premium,DL67543,Scratch on windows,good service,False,Unparseable timestamp(s)
788,RES-00789,C5926,CAR-08,Luxury,2026-01-31 12:07,NaN,2026-02-07 07:07,Bengaluru,12.966016,77.586999,...,3000,DISC15,150.0,3150.0,standard,INVALID,scratch on door,Good mileage,False,Unparseable timestamp(s)


In [24]:
df['Payment']

0          upi
1          UPI
2         card
3         card
4       wallet
         ...  
6995      cash
6996      card
6997       upi
6998       upi
6999      cash
Name: Payment, Length: 7000, dtype: str

In [25]:
df['Payment'] = df['Payment'].apply(cleaner.clean_payment)
df['Payment']

0          UPI
1          UPI
2         CARD
3         CARD
4       WALLET
         ...  
6995      CASH
6996      CARD
6997       UPI
6998       UPI
6999      CASH
Name: Payment, Length: 7000, dtype: str

In [26]:
df[["Odo_Start", "Odo_End"]]

,Odo_Start,Odo_End
0,30179,30857
1,41908,41829
2,100504,101678
3,66049,67079
4,41765,42647
...,...,...
6995,17666,17795
6996,25239,25558
6997,100059,100441
6998,99438,100933


In [27]:
results = df.apply(validator.validate_odometer, axis = 1)

df["odometer_valid"] = results.apply(lambda x: x[0])
df["odometer_error"] = results.apply(lambda x: x[1])

In [28]:
df[df["odometer_valid"] == False][["Odo_Start","Odo_End","odometer_error"]]

,Odo_Start,Odo_End,odometer_error
1,41908,41829,Odo_End (41829) < Odo_Start (41908)
15,73899,73816,Odo_End (73816) < Odo_Start (73899)
22,12781,12400,Odo_End (12400) < Odo_Start (12781)
46,30359,29976,Odo_End (29976) < Odo_Start (30359)
50,113324,113233,Odo_End (113233) < Odo_Start (113324)
...,...,...,...
6859,62317,61928,Odo_End (61928) < Odo_Start (62317)
6925,85035,84901,Odo_End (84901) < Odo_Start (85035)
6942,75999,75635,Odo_End (75635) < Odo_Start (75999)
6949,8975,8569,Odo_End (8569) < Odo_Start (8975)


In [29]:
df["odometer_valid"].value_counts()

odometer_valid
True     6656
False     344
Name: count, dtype: int64

In [30]:
# scenario 11 and 19

from analytics.transformer import compute_record_metrics

In [31]:
df[[
    "Odo_Start",
    "Odo_End",
    "Fuel_Level",
    "Rate",
    "Pickup_TS",
    "Return_TS"
]].head()

,Odo_Start,Odo_End,Fuel_Level,Rate,Pickup_TS,Return_TS
0,30179,30857,0.93,1500,2026-03-23 03:01,2026-03-24 18:01
1,41908,41829,0.90,800,2026-02-27 18:17,2026-03-01 14:17
2,100504,101678,0.94,2500,2026-02-28 17:07,2026-03-04 00:07
3,66049,67079,0.07,2500,2026-02-05 14:40,2026-02-09 12:40
4,41765,42647,0.40,1200,2026-02-18 08:59,2026-02-21 23:59


In [32]:
df_after = df.apply(
    lambda row: compute_record_metrics(row.to_dict()),
    axis=1
)

df_after = pd.DataFrame(df_after.tolist())

In [33]:
df_after[[
    "Odo_Start",
    "Odo_End",
    "Distance_km",
    "Fuel_Level",
    "Refuel_Event",
    "Rate",
    "Rate_Plan",
    "Cost_per_km"
]].head()

,Odo_Start,Odo_End,Distance_km,Fuel_Level,Refuel_Event,Rate,Rate_Plan,Cost_per_km
0,30179,30857,678,0.93,True,1500,standard,2.21
1,41908,41829,0,0.90,False,800,standard,0.00
2,100504,101678,1174,0.94,True,2500,eco,2.13
3,66049,67079,1030,0.07,False,2500,eco,2.43
4,41765,42647,882,0.40,False,1200,premium,1.36


In [34]:
records = df.to_dict(orient="records")
overlapping_ids = deduplicator.detect_overlapping_bookings(records)

print("Overlapping Reservations:", overlapping_ids)
print("Total overlaps detected:", len(overlapping_ids))

Overlapping Reservations: {'RES-03932', 'RES-01319', 'RES-01999', 'RES-06142', 'RES-00549', 'RES-03587', 'RES-05380', 'RES-03388', 'RES-03560', 'RES-06688', 'RES-05979', 'RES-03947', 'RES-05147', 'RES-01193', 'RES-04163', 'RES-06730', 'RES-02239', 'RES-00159', 'RES-04573', 'RES-02183', 'RES-04652', 'RES-03696', 'RES-05956', 'RES-05073', 'RES-05082', 'RES-06939', 'RES-00929', 'RES-06456', 'RES-01269', 'RES-02676', 'RES-04120', 'RES-03861', 'RES-01284', 'RES-06067', 'RES-00432', 'RES-05716', 'RES-00349', 'RES-00808', 'RES-01238', 'RES-06085', 'RES-03959', 'RES-00991', 'RES-06452', 'RES-06945', 'RES-03335', 'RES-03724', 'RES-03053', 'RES-04790', 'RES-02535', 'RES-01964', 'RES-01087', 'RES-04604', 'RES-02439', 'RES-00574', 'RES-02915', 'RES-04365', 'RES-04082', 'RES-02970', 'RES-06241', 'RES-06640', 'RES-01367', 'RES-02247', 'RES-02986', 'RES-05235', 'RES-05109', 'RES-00203', 'RES-06889', 'RES-06994', 'RES-01820', 'RES-04955', 'RES-05756', 'RES-05452', 'RES-04285', 'RES-02601', 'RES-06014'

In [35]:
df["Overlapping_Booking"] = df["Reservation_ID"].isin(overlapping_ids)

In [36]:
df["Overlapping_Booking"].value_counts()

Overlapping_Booking
True     6936
False      64
Name: count, dtype: int64

Interpretation:

The dataset reveals severe fleet scheduling conflicts. This demonstrates the importance of automated overlap detection in fleet management systems.

In [37]:
df["Vehicle_ID"].value_counts().head()

Vehicle_ID
CAR-07    255
CAR-14    251
CAR-03    248
CAR-06    248
CAR-28    246
Name: count, dtype: int64

In [38]:
df[["Notes"]].head(20)


,Notes
0,Scratch on windows
1,Bad Seats Quality
2,scratch on door
3,scratch on door
4,Speakers not working
5,Scratch on windows
6,Speakers not working
7,Contact Rahul 9876543210
8,Bad Seats Quality
9,scratch on door


In [39]:
df["Damage_Reported"] = df["Notes"].fillna("").apply(cleaner.detect_damage)

In [40]:
df["Damage_Reported"].value_counts()

Damage_Reported
False    4221
True     2779
Name: count, dtype: int64

In [41]:
df[["Notes","Damage_Reported"]].head(10)

,Notes,Damage_Reported
0,Scratch on windows,True
1,Bad Seats Quality,False
2,scratch on door,True
3,scratch on door,True
4,Speakers not working,False
5,Scratch on windows,True
6,Speakers not working,False
7,Contact Rahul 9876543210,False
8,Bad Seats Quality,False
9,scratch on door,True


In [59]:
df["Driver_License"].head(20)

0     INVAXXXXXXX
1     INVAXXXXXXX
2     DL12XXXXXXX
3     INVAXXXXXXX
4     DL12XXXXXXX
5     DL90XXXXXXX
6     DL90XXXXXXX
7     DL90XXXXXXX
8     DL67XXXXXXX
9     DL67XXXXXXX
10    DL90XXXXXXX
11    DL67XXXXXXX
12    DL90XXXXXXX
13    INVAXXXXXXX
14    DL12XXXXXXX
15    DL67XXXXXXX
16    DL90XXXXXXX
17    DL12XXXXXXX
18    DL12XXXXXXX
19    DL67XXXXXXX
Name: Driver_License, dtype: str

In [60]:
df["Driver_License"] = df["Driver_License"].apply(cleaner.mask_driver_license)
df['Driver_License'].head(10)

0    INVAXXXXXXX
1    INVAXXXXXXX
2    DL12XXXXXXX
3    INVAXXXXXXX
4    DL12XXXXXXX
5    DL90XXXXXXX
6    DL90XXXXXXX
7    DL90XXXXXXX
8    DL67XXXXXXX
9    DL67XXXXXXX
Name: Driver_License, dtype: str

Scenario 15 – Promo Code Validation

This rule verifies that promotional codes applied to reservations are valid and not expired.
It helps prevent revenue leakage caused by outdated or unauthorized promo codes.

In [44]:
df[["Reservation_ID","Promo_Code"]].head(10)

,Reservation_ID,Promo_Code
0,RES-00001,BOGUS99
1,RES-00002,NaN
2,RES-00003,BOGUS99
3,RES-00004,BOGUS99
4,RES-00005,BOGUS99
5,RES-00006,DISC15
6,RES-00007,BOGUS99
7,RES-00008,NaN
8,RES-00009,BOGUS99
9,RES-00010,SAVE10


In [45]:
promo_results = df.apply(validator.validate_promo_code, axis=1)
df["Promo_Valid"] = promo_results.apply(lambda x: x[0])
df["Promo_Error"] = promo_results.apply(lambda x: x[1])

df[[
    "Reservation_ID",
    "Promo_Code",
    "Promo_Valid",
    "Promo_Error"
]].head(10)

,Reservation_ID,Promo_Code,Promo_Valid,Promo_Error
0,RES-00001,BOGUS99,False,Invalid promo code: BOGUS99
1,RES-00002,NaN,False,Invalid promo code: NAN
2,RES-00003,BOGUS99,False,Invalid promo code: BOGUS99
3,RES-00004,BOGUS99,False,Invalid promo code: BOGUS99
4,RES-00005,BOGUS99,False,Invalid promo code: BOGUS99
5,RES-00006,DISC15,True,
6,RES-00007,BOGUS99,False,Invalid promo code: BOGUS99
7,RES-00008,NaN,False,Invalid promo code: NAN
8,RES-00009,BOGUS99,False,Invalid promo code: BOGUS99
9,RES-00010,SAVE10,True,


In [46]:
print("Total reservations:", len(df))
print("Invalid promo codes:", (df["Promo_Valid"] == False).sum())

Total reservations: 7000
Invalid promo codes: 3530


16.

In [47]:
df[["GPS_Lat","GPS_Lon"]].head(10)

,GPS_Lat,GPS_Lon
0,15.505045,85.735340
1,17.102400,80.326587
2,26.900000,81.057810
3,16.360000,86.378700
4,19.917400,87.728123
5,16.701430,72.632598
6,27.180000,76.668783
7,22.660000,80.736100
8,22.049100,85.280652
9,13.088400,78.639600


In [48]:
df["GPS_Lat"] = df["GPS_Lat"].apply(cleaner.smooth_gps)
df["GPS_Lon"] = df["GPS_Lon"].apply(cleaner.smooth_gps)

In [49]:
df[["GPS_Lat","GPS_Lon"]].head(10)

,GPS_Lat,GPS_Lon
0,15.5050,85.7353
1,17.1024,80.3266
2,26.9000,81.0578
3,16.3600,86.3787
4,19.9174,87.7281
5,16.7014,72.6326
6,27.1800,76.6688
7,22.6600,80.7361
8,22.0491,85.2807
9,13.0884,78.6396


17

In [50]:
df['Max_Speed_kmh'].head

<bound method NDFrame.head of 0       165 kmh
1            92
2            85
3            81
4            97
         ...   
6995    167 kmh
6996         49
6997         83
6998         53
6999         78
Name: Max_Speed_kmh, Length: 7000, dtype: str>

In [51]:
df['Max_Speed_kmh'] = df['Max_Speed_kmh'].apply(cleaner.normalize_speed)
df['Max_Speed_kmh'].head

<bound method NDFrame.head of 0       165
1        92
2        85
3        81
4        97
       ... 
6995    167
6996     49
6997     83
6998     53
6999     78
Name: Max_Speed_kmh, Length: 7000, dtype: int64>

18

In [52]:
df[["Notes","Customer_Feedback"]].head(10)

,Notes,Customer_Feedback
0,Scratch on windows,Good mileage
1,Bad Seats Quality,Good mileage
2,scratch on door,Good mileage
3,scratch on door,Smooth going
4,Speakers not working,Smooth going
5,Scratch on windows,Smooth going
6,Speakers not working,Smooth going
7,Contact Rahul 9876543210,Good mileage
8,Bad Seats Quality,email test@gmail.com
9,scratch on door,good service


In [53]:
df["Notes"] = df["Notes"].apply(cleaner.redact_pii)
df["Customer_Feedback"] = df["Customer_Feedback"].apply(cleaner.redact_pii)

In [54]:
df[["Notes","Customer_Feedback"]].head(10)

,Notes,Customer_Feedback
0,Scratch on windows,Good mileage
1,Bad Seats Quality,Good mileage
2,scratch on door,Good mileage
3,scratch on door,Smooth going
4,Speakers not working,Smooth going
5,Scratch on windows,Smooth going
6,Speakers not working,Smooth going
7,Contact Rahul [PHONE],Good mileage
8,Bad Seats Quality,email [EMAIL]
9,scratch on door,good service


In [55]:
df["Notes"].str.contains(r'\d{10}', na=False).sum()

np.int64(0)

20

In [56]:
df[["Reservation_ID","Rate","GST_Amount"]].head(10)

,Reservation_ID,Rate,GST_Amount
0,RES-00001,1500,75.0
1,RES-00002,800,NaN
2,RES-00003,2500,NaN
3,RES-00004,2500,125.0
4,RES-00005,1200,NaN
5,RES-00006,2000,100.0
6,RES-00007,2500,125.0
7,RES-00008,1000,50.0
8,RES-00009,1200,NaN
9,RES-00010,2000,100.0


In [57]:
gst_results = df.apply(validator.validate_gst, axis=1)
df["GST_Valid"] = gst_results.apply(lambda x: x[0])
df["GST_Error"] = gst_results.apply(lambda x: x[1])

In [58]:
df[[
    "Reservation_ID",
    "Rate",
    "GST_Amount",
    "GST_Valid",
    "GST_Error"
]].head(10)

,Reservation_ID,Rate,GST_Amount,GST_Valid,GST_Error
0,RES-00001,1500,75.0,True,
1,RES-00002,800,NaN,True,
2,RES-00003,2500,NaN,True,
3,RES-00004,2500,125.0,True,
4,RES-00005,1200,NaN,True,
5,RES-00006,2000,100.0,True,
6,RES-00007,2500,125.0,True,
7,RES-00008,1000,50.0,True,
8,RES-00009,1200,NaN,True,
9,RES-00010,2000,100.0,True,
